# Advanced Metaprogramming — Tutorial-Style Problems with Solutions

This notebook continues the same metaprogramming theme as the source material, but the problems are **new**.

The style is intentionally incremental. We will usually:

1. start with a concrete requirement,
2. write the simplest version,
3. inspect what Python actually created,
4. expose a subtle problem,
5. explain why it happens,
6. refine the metaclass or descriptor,
7. test the final behavior.

The goal is not just to end up with working code. The goal is to understand **when class creation happens, what a metaclass can change, and which hooks Python calls automatically**.

## What we will build

The problems cover three recurring applications of metaprogramming:

- generating value-like classes and descriptors,
- controlling instance creation and identity,
- generating configuration APIs from external data.

Along the way we will use `type.__new__`, metaclass `__call__`, descriptors, `__set_name__`, inheritance-aware metadata merging, registries, weak references, `inspect.Signature`, and `configparser`.

---

# Problem 1 — Why injecting descriptors *after* class creation can be too late

Suppose we want a metaclass to generate managed attributes for fields declared in `_fields`.

A descriptor can learn the attribute name it was assigned to through `__set_name__`.

That sounds perfect for generated fields. But there is a timing detail that is easy to miss.

First, let's build a descriptor that records the name Python gives it.

In [1]:
class NamedField:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        setattr(instance, self.storage_name, value)

When the descriptor is present in the class namespace **before** `type.__new__` creates the class, Python calls `__set_name__` automatically.

In [2]:
class ManualPoint:
    x = NamedField()
    y = NamedField()

ManualPoint.x.public_name, ManualPoint.y.storage_name

('x', '_y')

Now let's make the first metaclass attempt.

We will create the class first, then attach descriptors with `setattr`.

In [3]:
class LateDescriptorMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        for field in namespace.get('_fields', ()):
            setattr(cls, field, NamedField())
        return cls

class BrokenPoint(metaclass=LateDescriptorMeta):
    _fields = ('x', 'y')

Let's inspect one of the generated descriptors.

In [4]:
vars(BrokenPoint)['x'].__dict__

{}

There is no `public_name` or `storage_name`.

Why?

`type.__new__` scans the namespace it receives and invokes `__set_name__` on descriptors in that namespace. Our descriptor did not exist there yet. We attached it only after the class had already been created.

We *could* manually call `__set_name__`, but that duplicates behavior Python already knows how to perform.

A cleaner solution is to put generated descriptors into the class namespace **before** delegating to `type.__new__`.

In [5]:
class EarlyDescriptorMeta(type):
    def __new__(mcls, name, bases, namespace):
        namespace = dict(namespace)
        for field in namespace.get('_fields', ()):
            namespace[field] = NamedField()
        return super().__new__(mcls, name, bases, namespace)

class Point(metaclass=EarlyDescriptorMeta):
    _fields = ('x', 'y')

    def __init__(self, x, y):
        self.x = x
        self.y = y

Let's verify that `__set_name__` was called and that the descriptors work.

In [6]:
p = Point(10, 20)
(
    Point.x.public_name,
    Point.y.storage_name,
    p.x,
    p.y,
    vars(p),
)

('x', '_y', 10, 20, {'_x': 10, '_y': 20})

### Solution takeaway

For class-construction hooks such as `__set_name__`, **when** the metaclass injects an object matters.

If Python needs to process an object as part of class creation, put it in the namespace before calling `super().__new__`.

---

# Problem 2 — Generate validating fields from a schema

Our previous descriptor accepts any value.

Let's make the class declaration more expressive. We want this:

```python
class InventoryItem(metaclass=SchemaMeta):
    _schema = {
        'sku': str,
        'price': (int, float),
        'quantity': int,
    }
```

From that small schema, the metaclass should generate:

- one validating descriptor per field,
- slots for storage,
- an initializer,
- a useful representation.

We'll start with a descriptor that knows what types it accepts.

In [7]:
class TypedField:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f'{self.public_name} must be {self.expected_type!r}; '
                f'got {type(value).__name__}'
            )
        setattr(instance, self.storage_name, value)

Now the metaclass can translate the declarative schema into real class machinery.

Notice that `__slots__` also has to be placed into the namespace before class creation. Setting `__slots__` afterward does **not** retroactively change the instance layout.

In [8]:
class SchemaMeta(type):
    def __new__(mcls, name, bases, namespace):
        namespace = dict(namespace)
        schema = dict(namespace.get('_schema', {}))

        namespace['__slots__'] = tuple(f'_{field}' for field in schema)

        for field, expected_type in schema.items():
            namespace[field] = TypedField(expected_type)

        if '__init__' not in namespace:
            def __init__(self, *args, **kwargs):
                fields = tuple(schema)
                if len(args) > len(fields):
                    raise TypeError('too many positional arguments')

                values = dict(zip(fields, args))
                overlap = values.keys() & kwargs.keys()
                if overlap:
                    duplicate = next(iter(overlap))
                    raise TypeError(f'multiple values for {duplicate!r}')
                values.update(kwargs)

                missing = [field for field in fields if field not in values]
                extra = [key for key in values if key not in schema]
                if missing:
                    raise TypeError(f'missing fields: {missing}')
                if extra:
                    raise TypeError(f'unknown fields: {extra}')

                for field in fields:
                    setattr(self, field, values[field])
            namespace['__init__'] = __init__

        if '__repr__' not in namespace:
            def __repr__(self):
                pairs = ', '.join(
                    f'{field}={getattr(self, field)!r}' for field in schema
                )
                return f'{type(self).__name__}({pairs})'
            namespace['__repr__'] = __repr__

        return super().__new__(mcls, name, bases, namespace)

Let's create a class with almost no boilerplate.

In [9]:
class InventoryItem(metaclass=SchemaMeta):
    _schema = {
        'sku': str,
        'price': (int, float),
        'quantity': int,
    }

item = InventoryItem('A-17', price=12.5, quantity=4)
item

InventoryItem(sku='A-17', price=12.5, quantity=4)

The generated descriptors still run on every assignment, not only during construction.

In [10]:
item.quantity = 9
item.quantity

9

And invalid data should fail at the boundary.

In [11]:
try:
    item.price = 'expensive'
except TypeError as ex:
    print(type(ex).__name__ + ':', ex)

TypeError: price must be (<class 'int'>, <class 'float'>); got str


The slots should also prevent arbitrary new attributes.

In [12]:
try:
    item.discount = 0.1
except AttributeError as ex:
    print(type(ex).__name__ + ':', ex)

AttributeError: 'InventoryItem' object has no attribute 'discount' and no __dict__ for setting new attributes


### Solution takeaway

A metaclass can turn a compact declaration into multiple coordinated class features. The important part is to keep validation in the descriptor so every assignment follows the same rule.

---

# Problem 3 — Make the generated schema inheritance-aware

The previous `SchemaMeta` only looks at the schema declared in the current class.

What happens if a subclass wants to add fields?

In [13]:
class PersonV1(metaclass=SchemaMeta):
    _schema = {'name': str}

class EmployeeV1(PersonV1):
    _schema = {'employee_id': int}

Let's inspect the subclass schema and try to construct what we logically want.

In [14]:
EmployeeV1._schema

{'employee_id': int}

In [15]:
try:
    EmployeeV1(name='Ada', employee_id=101)
except TypeError as ex:
    print(type(ex).__name__ + ':', ex)

TypeError: unknown fields: ['name']


The subclass metaclass invocation saw only `employee_id` in its own namespace.

For inheritance-aware generation, we need to merge metadata from the base classes **before** generating the subclass methods and descriptors.

There is one more detail: slots inherited from a base class should not be re-declared in the subclass. The subclass only needs slots for newly introduced storage attributes.

In [16]:
class InheritedSchemaMeta(type):
    def __new__(mcls, name, bases, namespace):
        namespace = dict(namespace)

        inherited_schema = {}
        for base in bases:
            inherited_schema.update(getattr(base, '__schema__', {}))

        local_schema = dict(namespace.get('_schema', {}))
        full_schema = {**inherited_schema, **local_schema}
        namespace['__schema__'] = full_schema

        new_fields = [field for field in local_schema if field not in inherited_schema]
        namespace['__slots__'] = tuple(f'_{field}' for field in new_fields)

        for field, expected_type in local_schema.items():
            namespace[field] = TypedField(expected_type)

        def __init__(self, *args, **kwargs):
            fields = tuple(full_schema)
            if len(args) > len(fields):
                raise TypeError('too many positional arguments')

            values = dict(zip(fields, args))
            overlap = values.keys() & kwargs.keys()
            if overlap:
                duplicate = next(iter(overlap))
                raise TypeError(f'multiple values for {duplicate!r}')
            values.update(kwargs)

            missing = [field for field in fields if field not in values]
            extra = [field for field in values if field not in full_schema]
            if missing:
                raise TypeError(f'missing fields: {missing}')
            if extra:
                raise TypeError(f'unknown fields: {extra}')

            for field in fields:
                setattr(self, field, values[field])

        namespace['__init__'] = __init__

        def __repr__(self):
            values = ', '.join(
                f'{field}={getattr(self, field)!r}' for field in full_schema
            )
            return f'{type(self).__name__}({values})'

        namespace['__repr__'] = __repr__
        return super().__new__(mcls, name, bases, namespace)

Let's try the same hierarchy again.

In [17]:
class Person(metaclass=InheritedSchemaMeta):
    _schema = {'name': str, 'age': int}

class Employee(Person):
    _schema = {'employee_id': int, 'department': str}

emp = Employee('Ada', 37, 101, 'Research')
emp

Employee(name='Ada', age=37, employee_id=101, department='Research')

The merged schema should contain both base and subclass fields.

In [18]:
Employee.__schema__

{'name': str, 'age': int, 'employee_id': int, 'department': str}

And validation should still be applied through inherited and local descriptors.

In [19]:
try:
    emp.age = 'thirty-seven'
except TypeError as ex:
    print(type(ex).__name__ + ':', ex)

TypeError: age must be <class 'int'>; got str


A subclass can also override the expected type of an inherited field. Because the local schema is merged last, it wins.

In [20]:
class ExternalEmployee(Employee):
    _schema = {'employee_id': str}

ext = ExternalEmployee('Lin', 29, 'EXT-9', 'Consulting')
(ext, ExternalEmployee.__schema__)

(ExternalEmployee(name='Lin', age=29, employee_id='EXT-9', department='Consulting'),
 {'name': str, 'age': int, 'employee_id': str, 'department': str})

### Solution takeaway

Inheritance-aware metaprogramming usually requires two views of metadata:

- the **full effective metadata** used by generated behavior,
- the **local metadata** used to decide what must be newly installed in the subclass.

---

# Problem 4 — Generate a useful call signature without using `exec`

Our generated `__init__` accepts `*args, **kwargs`.

It works, but introspection is poor:

In [21]:
import inspect
inspect.signature(Employee)

<Signature (*args, **kwargs)>

For generated APIs, introspection matters. IDEs, documentation tools, and `help()` often rely on signatures.

We can attach an `inspect.Signature` to the generated class.

Let's create a metaclass derived from `InheritedSchemaMeta`. After the class exists, we can describe how it should appear to callers.

In [22]:
class SignatureSchemaMeta(InheritedSchemaMeta):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        parameters = [
            inspect.Parameter(
                field,
                kind=inspect.Parameter.POSITIONAL_OR_KEYWORD,
            )
            for field in cls.__schema__
        ]
        cls.__signature__ = inspect.Signature(parameters)
        return cls

Now let's define another generated value class.

In [23]:
class Coordinate(metaclass=SignatureSchemaMeta):
    _schema = {'latitude': (int, float), 'longitude': (int, float)}

inspect.signature(Coordinate)

<Signature (latitude, longitude)>

The runtime initializer is still the generic implementation, but tools asking for the class signature now see a precise interface.

In [24]:
c = Coordinate(42.6977, 23.3219)
(c, inspect.signature(Coordinate))

(Coordinate(latitude=42.6977, longitude=23.3219),
 <Signature (latitude, longitude)>)

We should also verify that the displayed signature and actual accepted fields stay synchronized.

In [25]:
assert tuple(inspect.signature(Coordinate).parameters) == tuple(Coordinate.__schema__)
print('signature matches schema')

signature matches schema


### Solution takeaway

Metaprogramming is not only about runtime behavior. A well-designed generated class should also expose useful metadata for humans and tools.

---

# Problem 5 — Build an automatic plugin registry with a metaclass

Another common pattern is a family of classes selected by a short key.

Without metaprogramming, every implementation might need to be manually added to a dictionary. That creates a second place to update whenever a new class is written.

We want subclasses to register themselves automatically:

```python
class JsonRenderer(Renderer):
    plugin_name = 'json'
```

Then `Renderer.create('json')` should construct the right implementation.

In [26]:
class RegistryMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        if not bases:
            cls._registry = {}
            return cls

        plugin_name = namespace.get('plugin_name')
        is_abstract = namespace.get('_abstract', False)

        if plugin_name is not None and not is_abstract:
            registry_owner = next(
                (base for base in cls.__mro__[1:] if hasattr(base, '_registry')),
                None,
            )
            registry = registry_owner._registry
            if plugin_name in registry:
                raise ValueError(f'duplicate plugin name: {plugin_name!r}')
            registry[plugin_name] = cls

        return cls

Now the base class can provide lookup and construction behavior.

In [27]:
class Renderer(metaclass=RegistryMeta):
    _abstract = True

    @classmethod
    def create(cls, plugin_name, *args, **kwargs):
        try:
            plugin_cls = cls._registry[plugin_name]
        except KeyError:
            available = ', '.join(sorted(cls._registry))
            raise KeyError(f'unknown renderer {plugin_name!r}; available: {available}')
        return plugin_cls(*args, **kwargs)

class JsonRenderer(Renderer):
    plugin_name = 'json'

    def render(self, data):
        import json
        return json.dumps(data, sort_keys=True)

class ReprRenderer(Renderer):
    plugin_name = 'repr'

    def render(self, data):
        return repr(data)

Let's inspect what registration did.

In [28]:
Renderer._registry

{'json': __main__.JsonRenderer, 'repr': __main__.ReprRenderer}

And now construction can be driven by data rather than `if/elif` chains.

In [29]:
renderer = Renderer.create('json')
renderer.render({'b': 2, 'a': 1})

'{"a": 1, "b": 2}'

A registry also needs to protect its key space. Let's intentionally try a duplicate.

In [30]:
try:
    class AnotherJsonRenderer(Renderer):
        plugin_name = 'json'
except ValueError as ex:
    print(type(ex).__name__ + ':', ex)

ValueError: duplicate plugin name: 'json'


### Solution takeaway

A registry metaclass turns **class definition itself** into the registration event. This removes a separate bookkeeping step and makes duplicate detection happen early.

---

# Problem 6 — Make singleton identity survive `copy` and pickle round-trips

A metaclass singleton usually intercepts normal construction through metaclass `__call__`.

But object copying and serialization can use different protocols.

Let's start with a basic singleton metaclass.

In [31]:
class BasicSingleton(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class AppState(metaclass=BasicSingleton):
    def __init__(self):
        self.mode = 'ready'

Normal construction returns the same object.

In [32]:
a = AppState()
b = AppState()
a is b

True

Now let's see what `copy.copy` does with the basic implementation.

In [33]:
import copy
copied = copy.copy(a)
(a is copied, id(a), id(copied))

(False, 1551189684784, 1551190313552)

The copy protocol can bypass the metaclass construction path.

If identity is part of the class contract, the class should participate in the copy protocol explicitly.

Pickle has a similar issue: we want unpickling to obtain the canonical singleton through the class call.

We can have the metaclass inject the relevant methods unless the class already defines custom ones.

In [34]:
class ProtocolSingleton(type):
    _instances = {}

    def __new__(mcls, name, bases, namespace):
        namespace = dict(namespace)

        namespace.setdefault('__copy__', lambda self: self)
        namespace.setdefault('__deepcopy__', lambda self, memo: self)

        if '__reduce__' not in namespace:
            def __reduce__(self):
                return (type(self), ())
            namespace['__reduce__'] = __reduce__

        return super().__new__(mcls, name, bases, namespace)

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

Let's use that metaclass for a cache object.

In [35]:
class GlobalCache(metaclass=ProtocolSingleton):
    def __init__(self):
        if not hasattr(self, 'data'):
            self.data = {}

cache = GlobalCache()
cache.data['answer'] = 42

Now shallow and deep copies should preserve identity.

In [36]:
(
    copy.copy(cache) is cache,
    copy.deepcopy(cache) is cache,
)

(True, True)

And a pickle round-trip should return the canonical instance too.

In [37]:
import pickle
restored = pickle.loads(pickle.dumps(cache))
(restored is cache, restored.data)

(True, {'answer': 42})

### Solution takeaway

Controlling `ClassName()` is not enough to control every object-creation pathway. Advanced singleton designs have to consider Python's copy and serialization protocols too.

---

# Problem 7 — Build a weak singleton that can disappear when nobody uses it

A permanent singleton cache keeps a strong reference to every singleton instance forever.

Sometimes that is exactly what we want. Sometimes it is not.

Suppose we want at most one live instance per class, but we are happy for that instance to be garbage-collected when the rest of the program stops using it.

`weakref.WeakValueDictionary` is a good fit. It stores values without keeping them alive by itself.

In [38]:
import weakref

class WeakSingleton(type):
    _instances = weakref.WeakValueDictionary()

    def __call__(cls, *args, **kwargs):
        instance = cls._instances.get(cls)
        if instance is None:
            instance = super().__call__(*args, **kwargs)
            cls._instances[cls] = instance
        return instance

class SessionIndex(metaclass=WeakSingleton):
    pass

As long as a strong reference exists, repeated construction should reuse the object.

In [39]:
s1 = SessionIndex()
s2 = SessionIndex()
s1 is s2

True

Now let's record the identity, remove our strong references, and force a garbage-collection cycle.

In [40]:
import gc
old_id = id(s1)
ref = weakref.ref(s1)
del s1, s2
gc.collect()
(ref() is None, list(WeakSingleton._instances.items()))

(True, [])

The weak registry no longer contains the object.

The next call may create a new instance because the old one no longer exists.

In [41]:
s3 = SessionIndex()
(ref() is None, id(s3) != old_id)

(True, True)

### Solution takeaway

A singleton policy does not have to mean “one object for the lifetime of the process.” Metaclass `__call__` can implement different identity lifetimes depending on how the instance registry stores references.

---

# Problem 8 — Layer a base INI file with an environment-specific override

Now let's return to configuration generation.

A common configuration pattern is:

- `base.ini` contains defaults,
- `prod.ini` overrides only the values that differ in production.

The generated API should expose the merged result through dot notation.

First we will create two tiny configuration files for the exercise.

In [42]:
from pathlib import Path

Path('base.ini').write_text(
    '[Database]\n'
    'host=localhost\n'
    'port=5432\n'
    'pool_size=5\n'
    '\n'
    '[Server]\n'
    'host=127.0.0.1\n'
    'port=8000\n',
    encoding='utf-8',
)

Path('prod.ini').write_text(
    '[Database]\n'
    'host=db.internal.example\n'
    'pool_size=20\n'
    '\n'
    '[Server]\n'
    'port=80\n',
    encoding='utf-8',
)

67

`configparser` can read multiple files in order. Later files override earlier values while unspecified values remain available.

In [43]:
import configparser

parser = configparser.ConfigParser()
parser.read(['base.ini', 'prod.ini'])
{
    section: dict(parser[section])
    for section in parser.sections()
}

{'Database': {'host': 'db.internal.example',
  'port': '5432',
  'pool_size': '20'},
 'Server': {'host': '127.0.0.1', 'port': '80'}}

Now let's turn each section into its own generated class.

We will use a tiny metaclass so the section's values become class attributes and its docstring explains where it came from.

In [44]:
class LayeredSectionMeta(type):
    def __new__(mcls, name, bases, namespace, *, section_name, items):
        namespace = dict(namespace)
        namespace['__doc__'] = f'Merged configuration for [{section_name}]'
        namespace['section_name'] = section_name
        namespace.update(dict(items))
        return super().__new__(mcls, name, bases, namespace)

The main metaclass will read the files, create one section class per section, and attach those generated section classes to the configuration class.

In [45]:
class LayeredConfigMeta(type):
    def __new__(mcls, name, bases, namespace, *, files):
        namespace = dict(namespace)

        parser = configparser.ConfigParser()
        loaded = parser.read(list(files))
        if len(loaded) != len(files):
            missing = [file for file in files if file not in loaded]
            raise FileNotFoundError(f'could not load: {missing}')

        namespace['sources'] = tuple(files)
        namespace['__doc__'] = f'Configuration layered from: {", ".join(files)}'

        for section_name in parser.sections():
            class_name = f'{section_name}Section'
            section_cls = LayeredSectionMeta(
                class_name,
                (object,),
                {},
                section_name=section_name,
                items=parser[section_name],
            )
            namespace[section_name.casefold()] = section_cls

        return super().__new__(mcls, name, bases, namespace)

Let's generate the production configuration class.

In [46]:
class ProductionConfig(
    metaclass=LayeredConfigMeta,
    files=('base.ini', 'prod.ini'),
):
    pass

Values present in `prod.ini` should override the base file.

In [47]:
(
    ProductionConfig.database.host,
    ProductionConfig.database.pool_size,
    ProductionConfig.server.port,
)

('db.internal.example', '20', '80')

But values missing from `prod.ini` should still come from `base.ini`.

In [48]:
(
    ProductionConfig.database.port,
    ProductionConfig.server.host,
)

('5432', '127.0.0.1')

Because these are real generated classes, introspection is useful too.

In [49]:
(
    ProductionConfig.__doc__,
    ProductionConfig.database.__doc__,
    vars(ProductionConfig.database),
)

('Configuration layered from: base.ini, prod.ini',
 'Merged configuration for [Database]',
 mappingproxy({'__doc__': 'Merged configuration for [Database]',
               'section_name': 'Database',
               'host': 'db.internal.example',
               'port': '5432',
               'pool_size': '20',
               '__module__': '__main__',
               '__dict__': <attribute '__dict__' of 'DatabaseSection' objects>,
               '__weakref__': <attribute '__weakref__' of 'DatabaseSection' objects>}))

### Solution takeaway

The metaclass does not need to implement merging itself. It can delegate file semantics to `configparser` and focus on transforming the resulting configuration into a convenient class API.

---

# Problem 9 — Detect configuration names that cannot safely become attributes

Generating attributes from external data has a naming problem.

INI keys can contain characters that are awkward for dot notation:

```ini
api-key = secret
```

We might normalize that to `api_key`.

But what if the file contains **both** `api-key` and `api_key`? A naive normalizer silently maps two different external keys to the same Python attribute.

Let's write a normalizer first.

In [50]:
import keyword
import re

def safe_identifier(name):
    normalized = re.sub(r'\W+', '_', name.strip().casefold())
    if not normalized:
        raise ValueError('empty identifier after normalization')
    if normalized[0].isdigit():
        normalized = '_' + normalized
    if keyword.iskeyword(normalized):
        normalized += '_'
    return normalized

[safe_identifier(name) for name in ['api-key', 'class', '2fa-mode']]

['api_key', 'class_', '_2fa_mode']

Now we'll make a section metaclass that keeps the original-to-Python mapping and rejects collisions.

In [51]:
class SafeSectionMeta(type):
    def __new__(mcls, name, bases, namespace, *, section_name, items):
        namespace = dict(namespace)
        keymap = {}
        reverse = {}

        for original_key, value in dict(items).items():
            attr_name = safe_identifier(original_key)
            if attr_name in reverse:
                other = reverse[attr_name]
                raise ValueError(
                    f'configuration keys {other!r} and {original_key!r} '
                    f'both normalize to {attr_name!r}'
                )
            reverse[attr_name] = original_key
            keymap[original_key] = attr_name
            namespace[attr_name] = value

        namespace['__keymap__'] = keymap
        namespace['section_name'] = section_name
        return super().__new__(mcls, name, bases, namespace)

Let's intentionally feed it conflicting keys.

In [52]:
try:
    SafeSectionMeta(
        'BrokenSection',
        (object,),
        {},
        section_name='API',
        items={'api-key': 'one', 'api_key': 'two'},
    )
except ValueError as ex:
    print(type(ex).__name__ + ':', ex)

ValueError: configuration keys 'api-key' and 'api_key' both normalize to 'api_key'


That failure is useful: it happens while the class is being built, instead of silently exposing the wrong value later.

Now let's use non-conflicting keys.

In [53]:
APISection = SafeSectionMeta(
    'APISection',
    (object,),
    {},
    section_name='API',
    items={'api-key': 'abc123', 'class': 'gold', '2fa-mode': 'required'},
)

(
    APISection.api_key,
    APISection.class_,
    APISection._2fa_mode,
    APISection.__keymap__,
)

('abc123',
 'gold',
 'required',
 {'api-key': 'api_key', 'class': 'class_', '2fa-mode': '_2fa_mode'})

### Solution takeaway

When metaprogramming turns external names into Python identifiers, normalization is part of the API design. Collision detection should be explicit and early.

---

# Problem 10 — Capstone: build a configuration class with generated documentation and a runtime override view

For the capstone, let's combine several ideas without simply merging all previous metaclasses together.

We want a generated configuration class that:

- reads layered INI files at class creation time,
- exposes sections through dot notation,
- records where values came from,
- provides a runtime `view(**overrides)` method,
- does **not** mutate the generated configuration class when a caller needs a temporary override.

First, let's create a small immutable section view for runtime use.

In [54]:
from types import MappingProxyType

class SectionView:
    __slots__ = ('_values',)

    def __init__(self, values):
        object.__setattr__(self, '_values', MappingProxyType(dict(values)))

    def __getattr__(self, name):
        try:
            return self._values[name]
        except KeyError:
            raise AttributeError(name) from None

    def __setattr__(self, name, value):
        raise AttributeError('SectionView is read-only')

    def __repr__(self):
        pairs = ', '.join(f'{k}={v!r}' for k, v in self._values.items())
        return f'SectionView({pairs})'

Next, the metaclass will preserve a plain dictionary snapshot in addition to generating section classes.

That snapshot gives `view()` a clean source of truth without having to reverse-engineer the generated classes.

In [55]:
class DocumentedConfigMeta(type):
    def __new__(mcls, name, bases, namespace, *, files):
        namespace = dict(namespace)

        parser = configparser.ConfigParser()
        loaded = parser.read(list(files))
        if len(loaded) != len(files):
            missing = [file for file in files if file not in loaded]
            raise FileNotFoundError(missing)

        snapshot = {
            section: dict(parser[section])
            for section in parser.sections()
        }

        namespace['__config_snapshot__'] = snapshot
        namespace['sources'] = tuple(files)
        namespace['__doc__'] = (
            f'{name}: generated from ' + ' -> '.join(files)
        )

        for section_name, values in snapshot.items():
            section_cls = type(
                f'{section_name}Section',
                (),
                {
                    '__doc__': f'[{section_name}] from layered sources',
                    **values,
                },
            )
            namespace[section_name.casefold()] = section_cls

        def view(cls, **overrides):
            merged = {
                section.casefold(): dict(values)
                for section, values in cls.__config_snapshot__.items()
            }

            for dotted_name, value in overrides.items():
                try:
                    section, key = dotted_name.split('__', 1)
                except ValueError:
                    raise ValueError(
                        f'override {dotted_name!r} must use section__key syntax'
                    ) from None

                if section not in merged:
                    raise KeyError(f'unknown section: {section!r}')
                if key not in merged[section]:
                    raise KeyError(f'unknown key: {section}.{key}')
                merged[section][key] = value

            return type(
                f'{cls.__name__}View',
                (),
                {
                    section: SectionView(values)
                    for section, values in merged.items()
                },
            )()

        namespace['view'] = classmethod(view)
        return super().__new__(mcls, name, bases, namespace)

Let's generate the class from the same layered files.

In [56]:
class AppConfig(
    metaclass=DocumentedConfigMeta,
    files=('base.ini', 'prod.ini'),
):
    pass

(AppConfig.__doc__, AppConfig.database.host, AppConfig.server.port)

('AppConfig: generated from base.ini -> prod.ini', 'db.internal.example', '80')

Now we can request a temporary runtime view.

We will use `section__key` for override names so a single keyword can identify both parts of the path.

In [57]:
local_test = AppConfig.view(
    database__host='127.0.0.1',
    server__port='9999',
)

(
    local_test.database.host,
    local_test.server.port,
)

('127.0.0.1', '9999')

The generated class itself should remain unchanged.

In [58]:
(
    AppConfig.database.host,
    AppConfig.server.port,
)

('db.internal.example', '80')

The runtime view is intentionally read-only.

In [59]:
try:
    local_test.server.port = '1234'
except AttributeError as ex:
    print(type(ex).__name__ + ':', ex)

AttributeError: SectionView is read-only


Bad override paths should fail clearly.

In [60]:
for bad in [
    {'missing__value': 'x'},
    {'server__missing': 'x'},
]:
    try:
        AppConfig.view(**bad)
    except (KeyError, ValueError) as ex:
        print(type(ex).__name__ + ':', ex)

KeyError: "unknown section: 'missing'"
KeyError: 'unknown key: server.missing'


### Capstone takeaway

Class-generation time and runtime do not have to compete.

The metaclass can create a rich, documented, inspectable baseline API once, while ordinary runtime objects provide temporary context-specific views without mutating that baseline.

---

# Final verification — run behavioral checks

A tutorial notebook is much more useful if the examples verify the contracts they claim to implement.

The following cell collects key assertions from all ten problems.

In [61]:
# Problem 1
assert Point.x.public_name == 'x'
assert Point(1, 2).y == 2

# Problem 2
assert repr(InventoryItem('X', 1.5, 2)) == "InventoryItem(sku='X', price=1.5, quantity=2)"

# Problem 3
assert tuple(Employee.__schema__) == ('name', 'age', 'employee_id', 'department')
assert ExternalEmployee.__schema__['employee_id'] is str

# Problem 4
assert tuple(inspect.signature(Coordinate).parameters) == ('latitude', 'longitude')

# Problem 5
assert Renderer.create('repr').render([1, 2]) == '[1, 2]'
assert Renderer._registry['json'] is JsonRenderer

# Problem 6
assert copy.copy(cache) is cache
assert copy.deepcopy(cache) is cache
assert pickle.loads(pickle.dumps(cache)) is cache

# Problem 7
assert SessionIndex() is s3

# Problem 8
assert ProductionConfig.database.host == 'db.internal.example'
assert ProductionConfig.database.port == '5432'

# Problem 9
assert APISection.__keymap__['api-key'] == 'api_key'
assert APISection.class_ == 'gold'

# Problem 10
assert local_test.server.port == '9999'
assert AppConfig.server.port == '80'

print('All tutorial checks passed.')

All tutorial checks passed.


# Additional solved mini-problems

The main tutorials were deliberately slow and incremental. The following mini-problems give you more lines of code and more patterns to study, but they are shorter.

## Mini-problem A — Count how many classes a metaclass has created

In [62]:
class CountingMeta(type):
    created = []

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        mcls.created.append(cls)
        return cls

class Alpha(metaclass=CountingMeta):
    pass

class Beta(metaclass=CountingMeta):
    pass

[class_.__name__ for class_ in CountingMeta.created]

['Alpha', 'Beta']

The state belongs to the metaclass itself, so it can observe all classes created through it.

## Mini-problem B — Reject reserved field names during class creation

In [63]:
class ReservedFieldMeta(InheritedSchemaMeta):
    reserved = {'save', 'delete'}

    def __new__(mcls, name, bases, namespace):
        local = set(namespace.get('_schema', {}))
        forbidden = local & mcls.reserved
        if forbidden:
            raise TypeError(f'reserved fields: {sorted(forbidden)}')
        return super().__new__(mcls, name, bases, namespace)

try:
    class BadRecord(metaclass=ReservedFieldMeta):
        _schema = {'save': str}
except TypeError as ex:
    print(type(ex).__name__ + ':', ex)

TypeError: reserved fields: ['save']


This demonstrates a useful policy: reject invalid class declarations immediately, while the class body is being converted into a class object.

## Mini-problem C — Add generated documentation for schema fields

In [64]:
class DocumentedSchemaMeta(SignatureSchemaMeta):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        lines = [f'{name} generated fields:']
        for field, expected in cls.__schema__.items():
            lines.append(f'- {field}: {expected!r}')
        cls.__doc__ = '\n'.join(lines)
        return cls

class Measurement(metaclass=DocumentedSchemaMeta):
    _schema = {'value': (int, float), 'unit': str}

print(Measurement.__doc__)

Measurement generated fields:
- value: (<class 'int'>, <class 'float'>)
- unit: <class 'str'>


## Mini-problem D — Store registry metadata, not just the class

In [65]:
class MetadataRegistryMeta(type):
    registry = {}

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        key = namespace.get('command_name')
        if key:
            mcls.registry[key] = {
                'class': cls,
                'description': namespace.get('description', ''),
            }
        return cls

class ExportCommand(metaclass=MetadataRegistryMeta):
    command_name = 'export'
    description = 'Export records to an external representation.'

MetadataRegistryMeta.registry

{'export': {'class': __main__.ExportCommand,
  'description': 'Export records to an external representation.'}}

## Mini-problem E — Observe the difference between class calls and instance calls

In [66]:
class TracingMeta(type):
    def __call__(cls, *args, **kwargs):
        print(f'metaclass __call__: constructing {cls.__name__}')
        return super().__call__(*args, **kwargs)

class CallableThing(metaclass=TracingMeta):
    def __call__(self, value):
        print(f'instance __call__: received {value}')
        return value * 2

thing = CallableThing()
thing(5)

metaclass __call__: constructing CallableThing
instance __call__: received 5


10

`CallableThing()` invokes the metaclass `__call__`. After the instance exists, `thing(5)` invokes the instance class's `__call__`. They are two different levels of the object model.

---

# Closing notes

The recurring design question in all of these examples is:

**At what stage should this behavior happen?**

- If it should happen while a class is being assembled, use class-creation machinery such as a metaclass `__new__`.
- If it should happen whenever a class is called to create an instance, metaclass `__call__` is the relevant hook.
- If it should happen when an instance attribute is read or written, descriptors are usually the better level.
- If generated APIs need to be pleasant to inspect, generate metadata such as docstrings and signatures too.

Metaprogramming becomes much easier to reason about once those stages are kept separate.